In [1]:
import pandas as pd
import numpy as np
import io

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from data_processing.geo_processing import (get_expression_data, 
                                            get_metadata, 
                                            get_platform,
                                            clean_metadata)

In [9]:
design_adni = pd.read_csv("./data/ADNI/design_with_real_target.tsv", sep="\t", index_col=0)
design_adni['Target'].value_counts()

Target
MCI        289
AD         233
Control    222
Name: count, dtype: int64

# Harvard Brain tissue resource center (HBTRC) AD & HD (GSE33000)

Abstract

Using expression profiles from postmortem prefrontal cortex samples of 624 dementia patients and non-demented controls, we investigated global disruptions in the co-regulation of genes in two neurodegenerative diseases, late-onset Alzheimer's disease (AD) and Huntington's disease (HD). We identified networks of differentially co-expressed (DC) gene pairs that either gained or lost correlation in disease cases relative to the control group, with the former dominant for both AD and HD and both patterns replicating in independent human cohorts of AD and aging. When aligning networks of DC patterns and physical interactions, we identified a 242-gene subnetwork enriched for independent AD/HD signatures. This subnetwork revealed a surprising dichotomy of gained/lost correlations among two inter-connected processes, chromatin organization and neural differentiation, and included DNA methyltransferases, DNMT1 and DNMT3A, of which we predicted the former but not latter as a key regulator. To validate the inter-connection of these two processes and our key regulator prediction, we generated two brain-specific knockout (KO) mice and show that Dnmt1 KO signature significantly overlaps with the subnetwork (P = 3.1 × 10(-12)), while Dnmt3a KO signature does not (P = 0.017).

Summary	

Dissecting the shared etiology of different diseases could benefit from a systematic search for associated molecules and their interactions. We investigated genome-wide disruptions in the co-regulation of genes in two neurodegenerative diseases, Alzheimer's or Huntington's disease (AD or HD), using expression profiles from postmortem prefrontal cortex samples of $624$ demented patients and non-demented control individuals with matched genotype and clinical data. A meta-analysis based screen for changes in coordinate expression patterns revealed differentially co-expressed (DC) gene pairs that either gained or lost correlation in disease cases relative to the control group, with the former being dominant for both AD and HD. Integration of disruptions common to AD and HD with large-scale data on protein-protein and protein-DNA interactions yielded a 242-gene sub-network that was enriched for proteins involved in neuronal differentiation and genetic associations to brain structural changes and dementia in subjects aged over 70 years. Replication of the AD DC network in independent human and mouse cohorts lends confidence to the comprehensive view we offer on dysregulated brain molecular pathways in AD and HD.
 	
Overall design	

DLPFC (BA9) brain tissues of AD patients, HD patients and non-demented controls samples were obtained from Harvard Brain tissue resource center (HBTRC). The HBTRC samples were primarily of Caucasian ancestry, as only eight non-Caucasian outliers were identified, and therefore excluded for further analysis. Post-mortem interval (PMI) was 17.8+8.3 hours (mean ± standard deviation), sample pH was 6.4±0.3 and RNA integrity number (RIN) was 6.8±0.8 for the average sample in the overall cohort. Tissues were profiled on a custom-made Agilent 44K array (GPL4372). 624 individual DLPFC samples were profiled against a common DLPFC pool constructed from the same set of samples.
### Read expression data and convert to gene symbols

In [2]:
matrix_path = "./data/GEO/GSE33000/GSE33000_series_matrix.txt"
platform_path = "./data/GEO/GSE33000/GPL4372.txt"

df_platform = get_platform(platform_path)
df_platform

,ID,EntrezGeneID,ORF,GB_ACC,PROBE_DERIVED_FROM_TRANSCRIPT,SPOT_ID,RosettaGeneModelID
0,10025930146,NaN,NaN,AA926913,non-public,NaN,HSG00215848
1,10025930335,NaN,NaN,AK021661,AK021661,NaN,HSG00264429
2,10025913794,439911.0,NaN,AL832060,AL832060,NaN,HSG00269218
3,10023807248,23443.0,SLC35A3,NM_012243,non-public,NaN,HSG00202393
4,10023809851,84056.0,KATNAL1,NM_001014380,non-public,NaN,HSG00284910
...,...,...,...,...,...,...,...
39297,10025908684,4939.0,OAS2,NM_016817,NM_016817,NaN,HSG00278151
39298,10023840430,81622.0,UNC93B1,NM_030930,NM_030930,NaN,HSG00272421
39299,10023812706,NaN,NaN,N70680,non-public,NaN,HSG00239400
39300,10023842346,158399.0,ZNF483,AB075842,AB075842,NaN,HSG00260976


In [3]:
df_expression = get_expression_data(matrix_path, df_platform)
df_expression

,GSM1423780,GSM1423781,GSM1423782,GSM1423783,GSM1423784,GSM1423785,GSM1423786,GSM1423787,GSM1423788,GSM1423789,...,GSM1424394,GSM1424395,GSM1424396,GSM1424397,GSM1424398,GSM1424399,GSM1424400,GSM1424401,GSM1424402,GSM1424403
A1BG,-0.066694,0.054097,0.025282,0.002485,-0.092606,-0.044708,-0.095566,-0.042258,-0.051480,-0.031068,...,0.055495,0.039800,0.06433,0.02736,-0.001750,0.02695,-0.012663,0.020015,0.00290,-0.05890
A2M,0.023919,-0.012131,0.001929,0.034625,0.019591,-0.029199,0.046953,0.133495,-0.003846,0.092455,...,-0.041550,-0.135200,-0.03280,-0.14095,-0.092100,0.05695,0.021630,-0.074950,-0.12550,0.01980
A2ML1,-0.031845,-0.030740,-0.085577,-0.066813,-0.030884,0.142922,0.288872,0.262960,-0.025880,0.067318,...,0.097150,0.169700,0.17085,0.07545,0.075405,0.01243,-0.070900,-0.051050,0.09445,0.05310
A3GALT2,0.015204,0.034282,-0.023002,-0.018289,0.020118,0.012233,0.020014,-0.075110,0.020017,-0.073667,...,0.037600,0.004340,0.06750,0.02300,0.090700,-0.01150,0.025400,-0.029400,0.02990,0.03020
A4GALT,0.089450,-0.006780,-0.314342,0.094836,-0.206281,0.206601,0.121640,0.389286,0.108989,0.230439,...,0.278000,0.154000,0.14200,-0.08640,0.021600,0.20000,0.034800,0.502000,0.12200,0.07130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZZEF1,0.106203,-0.040029,-0.146334,-0.000814,0.023013,-0.098126,0.016520,-0.075116,0.060029,0.098699,...,0.032100,-0.170000,-0.20300,-0.05700,-0.134000,-0.00908,0.173000,-0.054800,-0.27600,-0.13700
ZZZ3,0.063151,-0.118265,-0.027519,-0.250529,0.051360,0.089817,0.239130,0.072385,-0.058781,0.058170,...,0.129000,0.000186,-0.12000,0.24400,0.100000,-0.01880,-0.008470,-0.070500,0.05230,-0.00993
tcag7.1017,0.066667,0.096145,0.165024,-0.014006,0.054292,-0.236038,-0.114034,-0.320746,0.156975,-0.192163,...,-0.041200,0.018800,-0.07130,-0.01610,0.019700,-0.00392,0.146000,0.047200,0.05750,-0.04920
tcag7.216,0.060402,-0.007896,-0.008110,-0.238408,0.115933,-0.398103,0.000311,-0.235192,0.049596,0.096314,...,0.032050,-0.005325,0.03870,-0.09110,-0.047500,-0.02060,-0.357500,-0.104250,-0.05635,0.12350


#### Data Analysis

![UMAP Plot](data/GEO/GSE33000_ad_hd/umap.png)

![Volcano Plot](data/GEO/GSE33000_ad_hd/volcano.png)

![Mean Difference Plot](data/GEO/GSE33000_ad_hd/mean_difference.png)

### Read metadata and align smaples with labels and other info

In [4]:
df_metadata = get_metadata(matrix_path)
df_metadata

,characteristics_ch1,characteristics_ch1,characteristics_ch2,characteristics_ch2,characteristics_ch2,characteristics_ch2
GSM1423780,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 67 yrs,gender: female,disease status: Alzheimer's disease
GSM1423781,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 88 yrs,gender: male,disease status: Alzheimer's disease
GSM1423782,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 62 yrs,gender: male,disease status: Alzheimer's disease
GSM1423783,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 90 yrs,gender: female,disease status: Alzheimer's disease
GSM1423784,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 90 yrs,gender: female,disease status: Alzheimer's disease
...,...,...,...,...,...,...
GSM1424399,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 64 yrs,gender: male,disease status: Huntington's disease
GSM1424400,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 57 yrs,gender: female,disease status: Huntington's disease
GSM1424401,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 28 yrs,gender: male,disease status: Huntington's disease
GSM1424402,tissue: prefrontal cortex brain,sample type: reference,tissue: prefrontal cortex brain,age: 38 yrs,gender: male,disease status: Huntington's disease


In [5]:
df_meta_clean = clean_metadata(df_metadata)

conditions = [
    df_meta_clean['Disease status'].str.contains("Alzheimer's", case=False, na=False),
    df_meta_clean['Disease status'].str.contains("Huntington's", case=False, na=False)
]

# 2. Define the labels that match the conditions
choices = ['AD', 'HD']

# 3. Apply the logic (The 'default' handles everything else as Control)
df_meta_clean['Target'] = np.select(conditions, choices, default='Control')
df_meta_clean.to_csv("./data/GEO/GSE33000/GSE33000_meta.csv")
df_meta_clean

,Tissue,Sample type,Age,Gender,Disease status,Target
GSM1423780,prefrontal cortex brain,reference,67 yrs,female,Alzheimer's disease,AD
GSM1423781,prefrontal cortex brain,reference,88 yrs,male,Alzheimer's disease,AD
GSM1423782,prefrontal cortex brain,reference,62 yrs,male,Alzheimer's disease,AD
GSM1423783,prefrontal cortex brain,reference,90 yrs,female,Alzheimer's disease,AD
GSM1423784,prefrontal cortex brain,reference,90 yrs,female,Alzheimer's disease,AD
...,...,...,...,...,...,...
GSM1424399,prefrontal cortex brain,reference,64 yrs,male,Huntington's disease,HD
GSM1424400,prefrontal cortex brain,reference,57 yrs,female,Huntington's disease,HD
GSM1424401,prefrontal cortex brain,reference,28 yrs,male,Huntington's disease,HD
GSM1424402,prefrontal cortex brain,reference,38 yrs,male,Huntington's disease,HD


In [6]:
# df_nohd = df_meta_clean[df_meta_clean['Target']!= 'HD']
# df_nohd['Target']  = df_nohd['Target'].map({'AD':'Disease', 'Control':'Control'})
df_nohd = pd.read_csv("./data/GEO/GSE33000_ad_hd/GSE33000_meta_2cls.csv")
df_nohd[df_nohd['Target']=='Control']

,Unnamed: 0,Tissue,Sample type,Age,Gender,Disease status,Target
310,GSM1424090,prefrontal cortex brain,reference,56 yrs,female,non-demented,Control
311,GSM1424091,prefrontal cortex brain,reference,64 yrs,male,non-demented,Control
312,GSM1424092,prefrontal cortex brain,reference,95 yrs,female,non-demented,Control
313,GSM1424093,prefrontal cortex brain,reference,59 yrs,male,non-demented,Control
314,GSM1424094,prefrontal cortex brain,reference,53 yrs,male,non-demented,Control
...,...,...,...,...,...,...,...
462,GSM1424242,prefrontal cortex brain,reference,52 yrs,male,non-demented,Control
463,GSM1424243,prefrontal cortex brain,reference,61 yrs,male,non-demented,Control
464,GSM1424244,prefrontal cortex brain,reference,60 yrs,female,non-demented,Control
465,GSM1424245,prefrontal cortex brain,reference,55 yrs,female,non-demented,Control


In [174]:
df_ad_exp = df_expression[df_nohd.index.to_list()]

#df_ad_exp.to_csv("./data/GEO/GSE33000/GSE33000_AD_exp.csv")
df_ad_exp

,GSM1423780,GSM1423781,GSM1423782,GSM1423783,GSM1423784,GSM1423785,GSM1423786,GSM1423787,GSM1423788,GSM1423789,...,GSM1424237,GSM1424238,GSM1424239,GSM1424240,GSM1424241,GSM1424242,GSM1424243,GSM1424244,GSM1424245,GSM1424246
A1BG,-0.066694,0.054097,0.025282,0.002485,-0.092606,-0.044708,-0.095566,-0.042258,-0.051480,-0.031068,...,0.130492,0.125386,0.045632,0.144950,0.101484,-0.035529,0.075622,0.065244,0.054953,0.096101
A2M,0.023919,-0.012131,0.001929,0.034625,0.019591,-0.029199,0.046953,0.133495,-0.003846,0.092455,...,0.020941,0.029677,-0.070363,0.015219,0.007237,-0.067985,-0.010792,-0.087919,0.072740,0.022254
A2ML1,-0.031845,-0.030740,-0.085577,-0.066813,-0.030884,0.142922,0.288872,0.262960,-0.025880,0.067318,...,-0.047942,-0.008811,0.005295,-0.021817,-0.075660,-0.115346,-0.088835,-0.120331,-0.094495,-0.150833
A3GALT2,0.015204,0.034282,-0.023002,-0.018289,0.020118,0.012233,0.020014,-0.075110,0.020017,-0.073667,...,-0.016319,0.046531,0.005714,-0.019352,-0.018120,-0.045191,-0.021300,-0.038014,-0.028996,0.006306
A4GALT,0.089450,-0.006780,-0.314342,0.094836,-0.206281,0.206601,0.121640,0.389286,0.108989,0.230439,...,0.048791,-0.162938,-0.139816,-0.053166,-0.203363,-0.085769,-0.096128,0.068436,-0.039163,-0.034374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZZEF1,0.106203,-0.040029,-0.146334,-0.000814,0.023013,-0.098126,0.016520,-0.075116,0.060029,0.098699,...,-0.087733,-0.090970,-0.025511,-0.162615,-0.013019,-0.100798,0.017417,0.130757,0.083839,0.033530
ZZZ3,0.063151,-0.118265,-0.027519,-0.250529,0.051360,0.089817,0.239130,0.072385,-0.058781,0.058170,...,-0.070367,-0.058349,-0.116657,-0.112565,-0.029919,0.016300,-0.108027,0.016830,-0.073553,-0.136882
tcag7.1017,0.066667,0.096145,0.165024,-0.014006,0.054292,-0.236038,-0.114034,-0.320746,0.156975,-0.192163,...,0.091551,0.076485,0.025473,0.022584,0.021009,0.053928,0.072586,0.094532,0.106768,0.106998
tcag7.216,0.060402,-0.007896,-0.008110,-0.238408,0.115933,-0.398103,0.000311,-0.235192,0.049596,0.096314,...,0.032033,0.060010,-0.050296,0.031650,0.003122,-0.060836,0.090821,-0.294262,-0.118929,-0.029159


# Healthy Aging Expression data (GSE53890)

Abstract

Human neurons are functional over an entire lifetime, yet the mechanisms that preserve function and protect against neurodegeneration during ageing are unknown. Here we show that induction of the repressor element 1-silencing transcription factor (REST; also known as neuron-restrictive silencer factor, NRSF) is a universal feature of normal ageing in human cortical and hippocampal neurons. REST is lost, however, in mild cognitive impairment and Alzheimer's disease. Chromatin immunoprecipitation with deep sequencing and expression analysis show that REST represses genes that promote cell death and Alzheimer's disease pathology, and induces the expression of stress response genes. Moreover, REST potently protects neurons from oxidative stress and amyloid β-protein toxicity, and conditional deletion of REST in the mouse brain leads to age-related neurodegeneration. A functional orthologue of REST, Caenorhabditis elegans SPR-4, also protects against oxidative stress and amyloid β-protein toxicity. During normal ageing, REST is induced in part by cell non-autonomous Wnt signalling. However, in Alzheimer's disease, frontotemporal dementia and dementia with Lewy bodies, REST is lost from the nucleus and appears in autophagosomes together with pathological misfolded proteins. Finally, REST levels during ageing are closely correlated with cognitive preservation and longevity. Thus, the activation state of REST may distinguish neuroprotection from neurodegeneration in the ageing brain.

PMID:27851730

In [ ]:
def process_gds_soft(file_path):
    metadata_rows = []
    expression_start_line = 0
    
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            # get metadata
            if line.startswith('#GSM'):
                # split at the first '='
                parts = line.split(' = ', 1)
                gsm_id = parts[0].replace('#', '').strip()
                
                # Extract [Info] and [Source] from GEO GDS files "Value for GSMxxx: [Info]; src: [Source]"
                content = parts[1].split('Value for ' + gsm_id + ':')[-1].strip()
                
                if '; src: ' in content:
                    info, source = content.split('; src: ', 1)
                else:
                    info, source = content, ""
                
                metadata_rows.append({
                    'Sample': gsm_id, 
                    'Group_Info': info.strip(), 
                    'Source_Tissue': source.strip()
                })
            
            # find the start line of exp data
            if line.startswith('!dataset_table_begin'):
                expression_start_line = i + 1
                break

    df_all = pd.read_csv(file_path, sep='\t', skiprows=expression_start_line)
    
    # Clean up the end of the file
    if df_all.iloc[-1, 0].startswith('!'):
        df_all = df_all.iloc[:-1]

    # Find Gene symbols, look for "Gene symbol", "Gene Symbol", or "gene_symbol"
    potential_symbol_cols = ['Gene symbol', 'Gene Symbol', 'gene_symbol', 'IDENTIFIER']
    symbol_col = None
    for col in potential_symbol_cols:
        if col in df_all.columns:
            symbol_col = col
            break
    print(symbol_col)

    # separate Expression and Map
    gsm_cols = [c for c in df_all.columns if c.startswith('GSM')]
    df_expression = df_all[['ID_REF'] + gsm_cols].copy()
    
    if symbol_col:
        # Create mapping dictionary
        mapping = dict(zip(df_all['ID_REF'], df_all[symbol_col]))
        df_expression['ID_REF'] = df_expression['ID_REF'].map(mapping)
        
        # remove rows where gene symbol is missing or empty
        df_expression = df_expression.dropna(subset=['ID_REF'])
        df_expression = df_expression[df_expression['ID_REF'] != '']
        
        # Set gene symbols as index and convert to numeric
        df_expression.set_index('ID_REF', inplace=True)
        df_expression = df_expression.apply(pd.to_numeric, errors='coerce')
        
        # Merge duplicates: mean of probes for the same gene
        df_expression = df_expression.groupby(level=0).mean()

    df_metadata = pd.DataFrame(metadata_rows).set_index('Sample')
    df_metadata.index.name = None
    
    return df_expression, df_metadata

In [ ]:
def clean_gds_metadata(df_meta):
    """
    Specific cleaner for the '#GSM' metadata format in GDS files.
    """
    df = df_meta.copy()
    
    # Split "24 years old Male" into separate Age and Gender columns
    # This regex looks for 'years old' to split or common patterns
    if 'Characteristics' in df.columns:
        # Example split logic: "24 years old Male" -> ["24", "Male"]
        extracted = df['Characteristics'].str.extract(r'(\d+)\s*years\s*old\s*(.*)')
        df['Age'] = extracted[0]
        df['Gender'] = extracted[1]
        df.drop(columns=['Characteristics'], inplace=True)
        
    return df

In [ ]:
file_path = "./data/GEO/GDS_healthy_aging/GDS5204_full.soft"
df_healthy_exp, df_healthy_meta = process_gds_soft(file_path)

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_71330/3806961169.py:25: DtypeWarning: Columns (0: Platform_SPOTID) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(file_path, sep='\t', skiprows=expression_start_line)


In [156]:
#df_healthy_exp.to_csv("./data/GEO/GDS_healthy_aging/GDS5204_exp.csv")
df_healthy_exp

,GSM1303144,GSM1303147,GSM1303148,GSM1303151,GSM1303155,GSM1303145,GSM1303146,GSM1303149,GSM1303150,GSM1303152,...,GSM1303168,GSM1303171,GSM1303173,GSM1303176,GSM1303179,GSM1303180,GSM1303182,GSM1303181,GSM1303183,GSM1303184
A1BG,4.681620,4.777110,4.640340,4.603490,4.640320,4.43427,4.439940,4.551720,4.545890,4.620380,...,4.566090,4.600450,4.448560,4.407380,4.501740,4.537380,4.516900,4.151260,4.686730,4.557310
A1BG-AS1,5.315560,5.242890,5.143990,5.011950,5.108250,4.78550,5.099350,5.173560,4.957470,5.164730,...,4.813220,4.916110,5.192210,4.931450,5.156250,4.921690,5.117300,4.886260,5.001410,5.324110
A1CF,4.632150,4.477100,4.396135,4.457860,4.353430,4.52640,4.446510,4.592325,4.576180,4.506355,...,4.596230,4.809030,4.505475,4.779860,4.523290,4.538070,4.662775,4.478780,4.585330,4.945090
A2M,6.783250,6.318980,6.556460,6.566235,6.841790,6.43335,6.666985,6.733465,6.752655,6.758060,...,6.811085,7.114740,6.785545,7.011240,6.604970,6.811270,7.045235,6.188680,6.427420,6.778450
A2M-AS1,3.562480,3.722150,3.585640,3.778620,3.375500,3.91893,3.622140,3.608390,3.844470,3.758170,...,4.151650,4.050390,4.041840,4.215520,3.988000,4.236770,4.170170,4.008960,3.693930,4.128530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,4.808190,4.514550,4.631760,4.658950,4.658400,4.64654,4.514790,4.659250,4.514600,4.535040,...,4.618390,4.818360,4.523380,4.865310,4.628210,4.547220,4.870850,4.537460,4.927040,4.907530
ZYG11B,7.869983,7.242020,7.968693,7.920767,7.246670,7.76175,7.970550,7.666467,7.605997,7.480453,...,7.615420,6.277100,7.346220,6.824977,7.894630,6.907213,7.235617,7.164207,7.225693,6.248927
ZYX,6.439625,6.747385,7.060680,6.758875,6.661415,6.81748,6.223840,6.617920,6.734705,7.046140,...,6.904700,6.657630,7.288120,6.801100,7.379100,7.295810,7.026125,7.500130,7.281640,7.129215
ZZEF1,5.592280,5.661253,5.548063,5.697387,5.601960,5.57938,5.552660,5.546753,5.539913,5.635477,...,5.643923,5.653427,5.684153,5.688277,5.664407,5.663217,5.620943,5.672903,5.701787,5.349593


In [157]:
df_healthy_meta = clean_gds_metadata(df_healthy_meta)
df_healthy_meta = df_healthy_meta.rename_axis(None, axis=0)
df_healthy_meta['Target'] = ['Control'] * len(df_healthy_meta)
df_healthy_meta.to_csv("./data/GEO/GDS_healthy_aging/GDS5204_meta.csv")
df_healthy_meta

,Age,Gender,Target
GSM1303144,24,Male,Control
GSM1303147,26,Male_2,Control
GSM1303148,26,Male,Control
GSM1303151,29,Male,Control
GSM1303155,37,Male,Control
GSM1303145,25,Female_2,Control
GSM1303146,25,Female,Control
GSM1303149,27,Female,Control
GSM1303150,29,Female,Control
GSM1303152,33,Female,Control


# AD Hisayama study (GSE36980)
Summary	

To identify molecular pathological alterations in AD brains, we performed interspecies comparative microarray analyses using RNAs prepared from postmortem human brain tissues donated for the Hisayama study and hippocampal RNAs from the triple-transgenic mouse model of AD (3xTg-AD)
Three-way ANOVA of microarray data from frontal cortex, temporal cortex and hippocampus with presence/absence of AD and vascular dementia, and sex, as factors revealed that the gene expression profile is most significantly altered in the hippocampi of AD brains. Comparative analyses of the brains of AD patients and a mouse model of AD showed that genes involved in non-insulin dependent DM and obesity were significantly altered in both, as were genes related to psychiatric disorders and Alzheimer’s disease.
 	
Overall design	

We prepared RNA samples from the gray matter of frontal and temporal cortices and hippocampi derived from 88 postmortem brains, among which 26 cases were pathologically diagnosed as having AD or an AD-like disorder. High-quality RNA (RIN≧6.9) samples were subjected to microarray analysis using the Affymetrix Human Gene 1.0 ST platform, and only those results that passed examinations for quality assurance and quality control of the Human Gene 1.0 ST arrays were retrieved. In total, we obtained gene expression profiles from the following samples: 33 frontal cortex samples, among which 15 were from AD patients; 29 temporal cortex samples, among which 10 were from AD patients; 18 hippocampus samples, among which 8 were from AD patients

`attention`
This dataset has only 26 AD samples and 88-26 non-AD smaples. But from each sample, they took multiple tissues, each served as an individual sample
 	

In [146]:
file_path = "./data/GEO/GSE36980/GDS4758_full.soft"
df_exp, df_meta = process_gds_soft(file_path)
df_exp.to_csv("./data/GEO/GSE36980/GSE36980_exp.csv")
df_exp

,GSM907858,GSM907859,GSM907860,GSM907854,GSM907855,GSM907856,GSM907857,GSM907825,GSM907828,GSM907832,...,GSM907823,GSM907808,GSM907809,GSM907810,GSM907811,GSM907812,GSM907815,GSM907817,GSM907821,GSM907824
ID_REF,,,,,,,,,,,,,,,,,,,,,
1060P11.3///KIR2DS2///KIR2DL5B///KIR3DL3///KIR2DL5A///KIR3DS1///KIR3DL2///KIR3DL1///KIR2DS5///KIR2DS4///KIR2DS3///KIR2DS1///KIR2DL3///KIR2DL2///KIR2DL1///KIR3DS1///KIR3DL2///KIR3DL1///KIR2DS5///KIR2DS3///KIR2DL1,6.574380,6.622870,6.627220,6.33130,6.453100,6.570130,6.574130,6.456380,6.464880,6.479100,...,6.630380,6.655920,6.752610,6.729290,6.504830,6.573970,6.572920,6.528530,6.473440,6.282200
1060P11.3///KIR2DS2///KIR2DL5B///KIR3DL3///KIR2DL5A///KIR3DS1///KIR3DL2///KIR3DL1///KIR2DS5///KIR2DS4///KIR2DS3///KIR2DS1///KIR2DL4///KIR2DL3///KIR2DL2///KIR2DL1///KIR3DS1///KIR3DL2///KIR3DL1///KIR2DS5///KIR2DS3///KIR2DL1,6.615792,6.709540,6.450815,6.45410,6.603122,6.523122,6.607825,6.845632,6.477123,6.616928,...,6.622332,6.803263,6.750035,6.742565,6.487468,6.648327,6.654790,6.593407,6.523708,6.431598
1060P11.3///KIR2DS2///KIR3DL3///KIR2DL5A///KIR3DS1///KIR3DL2///KIR3DL1///KIR2DS5///KIR2DS4///KIR2DS3///KIR2DS1///KIR2DL3///KIR2DL2///KIR2DL1///KIR3DS1///KIR3DL2///KIR3DL1///KIR2DS5///KIR2DS3///KIR2DL1,6.620170,6.669905,6.527995,6.42862,6.519470,6.537455,6.637735,6.579375,6.378970,6.518515,...,6.586360,6.670760,6.624815,6.673640,6.460655,6.511205,6.519175,6.478545,6.417265,6.239215
A1BG,7.694970,7.748300,7.720650,7.67308,7.653720,7.664340,7.941390,7.728140,7.728150,7.802190,...,7.758610,8.134250,8.133830,8.140500,7.786780,7.972430,7.789470,7.829540,7.767220,7.520270
A1CF,6.132980,6.303350,6.192580,5.82571,5.974380,6.217550,6.398210,5.770490,5.770490,5.869530,...,5.904290,6.009970,5.847010,5.699960,5.790460,5.864500,5.902620,5.701270,5.588660,5.625580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,5.439650,5.698500,5.724590,5.64077,5.780600,5.824290,5.845640,5.496710,5.558930,5.511190,...,5.667800,5.806110,5.908110,5.891660,5.701510,5.993780,5.675350,5.645950,5.803830,5.552300
ZYG11B,10.171900,10.191500,10.108900,10.20100,9.918050,9.680410,9.531750,10.035800,10.254200,10.254200,...,10.180300,9.932580,10.060500,10.283900,10.373200,9.921720,9.920130,10.233800,9.911700,10.177200
ZYX,9.497170,9.237140,8.733430,9.22990,9.224740,9.071170,9.260430,9.552140,9.375850,9.619030,...,9.835230,9.703070,9.863970,9.707930,9.383360,9.736430,10.428800,9.912100,10.124500,9.969220


In [152]:
df_meta['Target'] = np.where(
    df_meta['Source_Tissue'].str.contains('non-AD', case=False, na=False), 
    'Control', 
    'Disease'
)
df_meta.to_csv("./data/GEO/GSE36980/GSE36980_meta.csv")
df_meta

,Group_Info,Source_Tissue,Target
GSM907858,"AD_HI, biological rep5",Hippocampus of of AD brain,Disease
GSM907859,"AD_HI, biological rep6",Hippocampus of of AD brain,Disease
GSM907860,"AD_HI, biological rep7",Hippocampus of of AD brain,Disease
GSM907854,"AD_HI, biological rep1",Hippocampus of of AD brain,Disease
GSM907855,"AD_HI, biological rep2",Hippocampus of of AD brain,Disease
...,...,...,...
GSM907812,"non-AD_FC, biological rep6",Frontal cortex of non-AD brain,Control
GSM907815,"non-AD_FC, biological rep9",Frontal cortex of non-AD brain,Control
GSM907817,"non-AD_FC, biological rep11",Frontal cortex of non-AD brain,Control
GSM907821,"non-AD_FC, biological rep15",Frontal cortex of non-AD brain,Control


# AD various severity (GSE1297)
Summary	

Expression profiling of brain hippocampi from 22 postmortem subjects with Alzheimer's disease (AD) at various stages of severity. 7, 8, and 7 subjects diagnosed with incipient, moderate, and severe AD respectively. 

For these data, we analyzed hippocampal gene expression of nine control and 22 AD subjects of varying severity on 31 separate microarrays. We then tested the correlation of each gene's expression with MiniMental Status Examination (MMSE) and neurofibrillary tangle (NFT) scores across all 31 subjects regardless of diagnosis. These tests revealed a major transcriptional response comprising thousands of genes significantly correlated with AD markers. Several hundred of these genes were also correlated with AD markers across only control and incipient AD subjects (MMSE > 20).



In [11]:
df_exp, df_meta = process_gds_soft("./data/GEO/GSE1297/GDS810_full.soft")
df_exp.to_csv("./data/GEO/GSE1297/GDS810_exp.csv")
df_exp

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_40871/3193815925.py:34: DtypeWarning: Columns (0: Platform_SPOTID) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(file_path, sep='\t', skiprows=expression_start_line)


Gene symbol


,GSM21215,GSM21217,GSM21218,GSM21219,GSM21220,GSM21221,GSM21226,GSM21231,GSM21232,GSM21204,...,GSM21224,GSM21227,GSM21230,GSM21203,GSM21206,GSM21207,GSM21208,GSM21212,GSM21213,GSM21229
ID_REF,,,,,,,,,,,,,,,,,,,,,
A1CF,323.900000,240.6,272.70,374.900000,423.100000,359.900000,506.9,338.50,405.800000,468.200000,...,229.600000,385.600000,469.900000,505.300000,312.400000,818.800000,608.600000,511.90,741.700000,247.700000
A2M,4116.900000,2922.2,2298.30,4792.800000,3319.400000,4018.600000,3256.6,3800.80,4217.000000,3389.100000,...,5099.100000,4174.400000,6217.200000,2165.700000,3852.800000,1090.000000,3031.400000,3112.40,3836.300000,3356.300000
A4GALT,51.900000,15.5,31.70,54.100000,53.700000,101.100000,51.4,120.50,61.800000,161.500000,...,104.000000,56.800000,118.100000,45.400000,44.400000,239.500000,115.200000,119.30,41.200000,38.100000
A4GNT,155.500000,184.2,186.10,257.900000,136.000000,205.200000,360.3,165.90,255.500000,251.500000,...,239.600000,350.500000,222.400000,344.300000,179.700000,320.200000,265.500000,306.00,334.200000,210.500000
AAAS,40.700000,66.3,123.10,190.800000,223.600000,60.300000,258.8,110.70,101.100000,28.200000,...,230.200000,187.200000,65.200000,42.600000,45.100000,42.700000,45.200000,170.60,65.600000,121.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZXDB///ZXDA,51.000000,52.9,49.40,38.900000,57.150000,12.500000,84.8,79.75,82.300000,57.350000,...,55.050000,59.700000,74.550000,86.550000,60.850000,72.450000,95.000000,85.60,83.400000,43.250000
ZXDC,141.100000,52.6,120.90,160.700000,81.100000,51.500000,24.9,73.40,93.900000,345.000000,...,230.900000,183.800000,315.000000,78.600000,36.300000,349.300000,31.200000,182.20,121.500000,236.600000
ZYX,415.050000,279.0,934.25,984.400000,585.900000,614.300000,493.6,872.35,708.650000,379.800000,...,689.750000,517.550000,763.750000,420.450000,699.150000,858.050000,323.750000,587.25,435.100000,654.700000


In [15]:
df_meta['Group_Info'] = df_meta['Group_Info'].str.split().str[0]
df_meta['Target'] = np.where(df_meta['Group_Info'].str.strip() == 'Control',
                             'Control',
                             'Disease')
df_meta.to_csv("./data/GEO/GSE1297/GDS810_meta.csv")
df_meta

,Group_Info,Source_Tissue,Target
GSM21215,Control,hippocampal CA1 tissue,Control
GSM21217,Control,hippocampal CA1 tissue,Control
GSM21218,Control,hippocampal CA1 tissue,Control
GSM21219,Control,hippocampal CA1 tissue,Control
GSM21220,Control,hippocampal CA1 tissue,Control
GSM21221,Control,hippocampal CA1 tissue,Control
GSM21226,Control,hippocampal CA1 tissue,Control
GSM21231,Control,hippocampal CA1 tissue,Control
GSM21232,Control,hippocampal CA1 tissue,Control
GSM21204,Incipient,hippocampal CA1 tissue,Disease


# AD blood Mononuclear (GSE4226)
Overall design	

Peripheral blood mononuclear cells were obtained from normal elderly control (NEC) and Alzheimer disease (AD) subjects. Targets from biological replicates of female (n=7) and male (n=7) NEC and female (n=7) and male (n=7) AD were generated and the expression profiles were determined using the NIA Human MGC cDNA microarray.

Summary	

Comparisons between the sample groups (normal elderly control (NEC) and Alzheimer disease (AD)) allow the identification of genes with disease and gender expression patterns.


In [20]:
df_exp, df_meta = process_gds_soft("./data/GEO/GSE4226_blood/GDS2601_full.soft")
df_exp.to_csv("./data/GEO/GSE4226_blood/GSE4226_exp.csv")
df_exp

Gene symbol


,GSM96477,GSM96478,GSM96479,GSM96480,GSM96481,GSM96482,GSM96483,GSM96462,GSM96463,GSM96464,...,GSM96474,GSM96475,GSM96476,GSM96454,GSM96455,GSM96456,GSM96457,GSM96458,GSM96459,GSM96461
ID_REF,,,,,,,,,,,,,,,,,,,,,
A4GALT,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AAAS,5.495740,6.302660,5.942080,4.883270,6.848760,3.674490,4.450150,5.386630,3.095020,4.029010,...,6.663950,4.005710,4.774210,2.620080,4.127160,3.900380,3.619730,3.167260,3.094820,3.843800
AAK1,2.905990,3.639750,2.615090,3.353390,2.987430,2.926970,3.410220,1.445600,1.481930,2.251140,...,3.704270,3.411130,1.630540,1.080580,4.215000,2.121790,1.955890,1.497930,1.575290,1.043270
AAMDC,1.932679,2.741176,1.815826,2.155920,2.750144,1.249338,2.955672,2.401512,2.929576,1.812407,...,0.144761,1.653955,1.978897,1.096273,3.809789,1.023046,1.160068,2.544209,2.509430,1.744825
AAR2,0.565454,0.582946,0.462196,0.441741,0.308593,0.328576,0.159352,0.672728,0.344044,0.632389,...,0.159985,0.318665,0.324556,0.527273,0.282913,0.527822,0.507537,0.477550,0.439516,0.525636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSWIM1,13.485800,12.878200,11.567800,7.545330,12.758600,7.553610,10.095200,0.954236,0.801625,0.932088,...,13.599600,14.395700,24.734500,1.253540,38.459300,0.776908,0.700314,0.634061,0.961219,0.737860
ZUFSP,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.095713,0.127616,...,0.000000,0.485714,0.964518,0.000000,4.631680,0.000000,0.000000,0.000000,0.000000,0.000000
ZWINT,0.000000,0.000000,0.121955,0.000000,0.000000,0.579677,0.156980,0.000000,0.000000,0.000000,...,0.000000,0.015974,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
df_meta['Target'] = np.where(
    df_meta['Group_Info'].str.contains('AD'),
    'Disease',
    'Control'
)
df_meta.to_csv("./data/GEO/GSE4226_blood/GSE4226_meta.csv")
df_meta

,Group_Info,Source_Tissue,Target
GSM96477,ADR386M,peripheral blood,Disease
GSM96478,AD2R622M,peripheral blood,Disease
GSM96479,AD906M,peripheral blood,Disease
GSM96480,AD911M,peripheral blood,Disease
GSM96481,AD933M,peripheral blood,Disease
GSM96482,AD1022M,peripheral blood,Disease
GSM96483,ADR1032M,peripheral blood,Disease
GSM96462,AD2R411F,peripheral blood,Disease
GSM96463,ADR430F,peripheral blood,Disease
GSM96464,AD940F,peripheral blood,Disease


# AD Various Severity (GSE28146)
Overall design

+ Samples: Control & three stages of AD (incipient, moderate, severe)
+ Laser Capture Microdissection (LCM) to get CA1 hippocampal gray matter
+ Collect these precise tissue and extract RNA
+ Sequencing

Summary	

Alzheimer's disease (AD) is a devastating neurodegenerative disorder that threatens to reach epidemic proportions as our population ages. Although much research has examined molecular pathways associated with AD, relatively few studies have focused on critical early stages. Our prior microarray study correlated gene expression in human hippocampus with AD markers. Results suggested a new model of early-stage AD in which pathology spreads along myelinated axons, orchestrated by upregulated transcription and epigenetic factors related to growth and tumor suppression (Blalock et al., 2004). However, the microarray analyses were performed on RNA from fresh frozen hippocampal tissue blocks containing both gray and white matter, potentially obscuring region-specific changes. In the present study, we used laser capture microdissection to exclude major white matter tracts and selectively collect CA1 hippocampal gray matter from formalin-fixed, paraffin-embedded (FFPE) hippoc ampal sections of the same subjects assessed in our prior study. Microarray analyses of this gray matter-enriched tissue revealed many correlations similar to those seen in our prior study, particularly for neuron-related genes. Nonetheless, in the laser-captured tissue, we found a striking paucity of the AD-associated epigenetic and transcription factor genes that had been strongly overrepresented in the prior tissue block study. In addition, we identified novel pathway alterations that may have considerable mechanistic implications, including downregulation of genes stabilizing ryanodine receptor Ca2+ release and upregulation of vascular development genes. We conclude that FFPE tissue can be a reliable resource for microarray studies, that upregulation of growth-related epigenetic/ transcription factors with incipient AD is predominantly localized to white matter, further supporting our prior findings and model, and that alterations in vascular and ryanodine receptor-relat ed pathways in gray matter are closely associated with incipient AD.
 	

In [30]:
df_exp, df_meta = process_gds_soft("./data/GEO/GSE28146_laser_vary_severity/GDS4136_full.soft")
df_exp.to_csv("./data/GEO/GSE28146_laser_vary_severity/GSE28146_exp.csv")
df_exp

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_40871/3193815925.py:34: DtypeWarning: Columns (0: Platform_SPOTID) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(file_path, sep='\t', skiprows=expression_start_line)


Gene symbol


,GSM697332,GSM697312,GSM697327,GSM697334,GSM697336,GSM697309,GSM697311,GSM697328,GSM697326,GSM697330,...,GSM697320,GSM697310,GSM697333,GSM697337,GSM697335,GSM697314,GSM697317,GSM697313,GSM697322,GSM697316
ID_REF,,,,,,,,,,,,,,,,,,,,,
A1BG,164.400000,612.300000,45.500000,14.900000,56.30,95.600000,11.000000,126.200000,26.500000,89.800000,...,30.400000,158.300000,28.100000,20.400000,67.800000,21.3,131.500000,103.400000,63.300000,29.300000
A1BG-AS1,37.000000,44.500000,92.700000,106.800000,202.50,151.800000,101.800000,44.000000,125.900000,322.900000,...,307.800000,30.400000,36.900000,135.700000,171.000000,226.1,87.800000,194.400000,466.600000,110.700000
A1CF,88.600000,177.500000,95.250000,113.250000,250.05,101.800000,129.250000,86.950000,11.150000,62.850000,...,33.950000,97.600000,42.200000,46.450000,253.650000,81.2,49.900000,33.800000,74.600000,36.700000
A2M,848.650000,528.950000,845.600000,473.650000,1053.55,353.800000,436.600000,970.200000,367.150000,1084.400000,...,743.700000,632.450000,611.450000,658.700000,640.500000,506.7,1106.050000,424.400000,708.200000,764.900000
A2M-AS1,1500.100000,631.000000,1231.500000,776.900000,327.70,379.800000,840.200000,569.600000,825.100000,748.700000,...,459.900000,470.300000,1000.100000,817.600000,777.800000,390.7,851.700000,566.300000,497.800000,871.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,39.200000,61.400000,139.200000,18.400000,33.80,11.200000,36.800000,10.800000,7.800000,7.400000,...,122.500000,181.300000,21.300000,18.400000,18.300000,9.8,24.400000,20.000000,16.800000,13.400000
ZYG11B,1254.566667,1295.633333,1317.133333,891.233333,796.00,1778.433333,1442.000000,1521.100000,1451.866667,1226.833333,...,1073.433333,1637.666667,913.266667,1627.433333,983.400000,1378.6,821.033333,1977.266667,1654.566667,1341.966667
ZYX,750.050000,959.700000,389.450000,323.700000,555.85,511.800000,703.300000,620.450000,383.600000,788.300000,...,392.550000,728.700000,863.400000,463.550000,588.550000,393.3,594.050000,307.550000,440.000000,480.000000


In [29]:
df_meta['Group_Info'] = df_meta['Group_Info'].str.split().str[0]
df_meta['Target'] = np.where(
    df_meta['Group_Info'].str.strip() =='Control',
    'Control',
    'Disease'
)
df_meta.to_csv("./data/GEO/GSE28146_laser_vary_severity/GSE28146_meta.csv")
df_meta

,Group_Info,Source_Tissue,Target
GSM697332,Severe,laser capture CA1 tissue gray matter from form...,Disease
GSM697312,Control,laser capture CA1 tissue gray matter from form...,Control
GSM697327,Moderate,laser capture CA1 tissue gray matter from form...,Disease
GSM697334,Severe,laser capture CA1 tissue gray matter from form...,Disease
GSM697336,Severe,laser capture CA1 tissue gray matter from form...,Disease
GSM697309,Control,laser capture CA1 tissue gray matter from form...,Control
GSM697311,Control,laser capture CA1 tissue gray matter from form...,Control
GSM697328,Moderate,laser capture CA1 tissue gray matter from form...,Disease
GSM697326,Moderate,laser capture CA1 tissue gray matter from form...,Disease
GSM697330,Moderate,laser capture CA1 tissue gray matter from form...,Disease
